# Cluster output
- cluster the top/worst settings
- embed output 
- use BERTopic
- save representative docs to be passed to LLM later
- run inference on protocol codes to see what topic it aligns with
- calculate percent coverage/relevance with new codes

In [ ]:
import nltk

import pandas as pd
# nltk.download("stopwords")
import sys
sys.path.append("../")

from sklearn.metrics import silhouette_score
from travail_code import vis_documents
from travail_code import llm_topic as lt
import pickle

import importlib
importlib.reload(vis_documents)
importlib.reload(lt)

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoConfig
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
import nltk
import numpy as np
import torch
torch.manual_seed(28)
np.random.seed(28)

In [ ]:
# load in files
path = 
file = 

output = pd.read_csv(f"{path}/{file}", delimiter = "\t")

## Select one group

In [ ]:
select_ids = [
('meta-llama/Llama-3.1-8B-Instruct', # best l8b
         'base_c',
        'race_anthro',
               'race',
     'chunk_paired'),
     ('meta-llama/Llama-3.1-8B-Instruct', # worst l8b
              'cot_t',
    'specified_anthro',
                 'race',
      'questions_chunk'),
('meta-llama/Llama-3.2-1B-Instruct', # best l1b
              'base_c',
    'specified_anthro',
                 'base',
      'questions_chunk'),
('meta-llama/Llama-3.2-1B-Instruct', # worst l1b
            'base_c',
            'race_anthro',
            'base',
            'full')
]

id_num = 0

uid = ["model_name", "prompt_name", 
        "identity", "context", "data_proc"]
sub_output = output[output[uid].apply(tuple, axis=1) == select_ids[id_num]]
display(sub_output.shape)

# parse out output:
exp_df = lt.parse_outputs(sub_output, 'output')
exp_df.dropna(subset = 'code', inplace = True)
exp_df = exp_df[exp_df['code']!=""]
print(exp_df.shape)
exp_df['examples'] = exp_df['examples'].str.strip('**|:|\n').str.strip()

gdupes =  exp_df.groupby('code').size().sort_values(ascending=False)
print('num_max_duplicates', gdupes.max())
print(len(gdupes))

## drop duplicates
group = exp_df.drop_duplicates(subset="code", ignore_index=True)
display(group.shape)

codes = group['code']
# count number of words per code
num_words = codes.str.split().apply(lambda x: len(x))
print(num_words.mean(),num_words.std())

In [ ]:
# calculate number of hours to complete this setting
sub_output['time_elapsed'].sum()/3600

In [ ]:
# how many times did duplicate codes appear over n times
display(gdupes[gdupes > 30])
len(gdupes[gdupes==1])

## Start BERTopic
- Define BERTopic settings
- Embed output codes

In [ ]:
# define models

model = "all-mpnet-base-v2"  #highest avg performance
    # "all-distilroberta-v1"
    # "all-MiniLM-L6-v2"
    # HKunlp/instructor 
    # intfloat/e5-mistral-7binstruct 
config = AutoConfig.from_pretrained(f'sentence-transformers/{model}')
stopwords = nltk.corpus.stopwords.words('english')

# embedding model
embedding_model = SentenceTransformer(model) # topic tokenizer

# Tokenize topics
vectorizer_model = CountVectorizer(stop_words= stopwords, 
                                   min_df = 3,
                                ngram_range = (1,3)
                                   )
# Create topic representation
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words = True)

In [ ]:
# # get embeddings and never do it again
codes = list(group['code'])
embeddings = embedding_model.encode(codes, batch_size = 8)


## gridsearch parameters to get topic model with highest silhouette scores

In [ ]:
# gridsearch parameter
n_neighbors = [20,40,60]
n_components = [2,5,15]
min_cluster_size = [20,40,60]

import itertools

sil_dict = {}
for nn, nc, mcs in itertools.product(n_neighbors, n_components, min_cluster_size):
    print(f"n_neighbors: {nn}, n_components: {nc}, min_cluster_size: {mcs}")
    # dimensionality reduction 
    umap_model = UMAP(n_neighbors=nn, n_components=nc, 
        min_dist=0.0, metric='cosine',
        random_state = 28) 
    # Cluster reduced embeddings
    hdbscan_model = HDBSCAN(min_cluster_size=nn, metric='euclidean', 
        cluster_selection_method='eom', prediction_data=True)
    reduced_embeddings = umap_model.fit_transform(embeddings) # for visualization (to avoid running encoding again)
    # overall topic model pipeline
    topic_model = BERTopic(
        # embedding_model=embedding_model,          # Step 1 - Extract embeddings
        umap_model=umap_model,                    # Step 2 - Reduce dimensionality
        hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
        vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
        ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words)
        calculate_probabilities=True
    )
    topics, probs = topic_model.fit_transform(codes,  
                                            embeddings = embeddings) # return assignment of topics per doc
    print(len(topic_model.get_topic_info()))
    if len(topic_model.get_topic_info()) >1:
        indices = [index for index, topic in enumerate(topics) if topic != -1]
        X = reduced_embeddings[np.array(indices)]
        labels = [topic for _, topic in enumerate(topics) if topic != -1]
        # get silhouette score ( measuring cluster quality when the clusters are convex-shaped -- how similar object is to own cluster vs others
        #  may not perform well if the data clusters have irregular shapes or are of varying sizes)
        score = silhouette_score(X, labels)
        print(score)
        sil_dict[(nn, nc, mcs)] = score  # max of 1, min of -1 (.7 = strong)

In [ ]:
print([i for i in sil_dict.keys() if sil_dict[i] == max(sil_dict.values())])
print(max(sil_dict.values()))

## Run best bertopic

In [ ]:
## get best model
nn=20
nc = 5
mcs= 40
umap_model = UMAP(n_neighbors= nn , n_components=nc,
min_dist=0.0, metric='cosine',
random_state = 28) # for visualization
# Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=mcs, metric='euclidean', 
    cluster_selection_method='eom', prediction_data=True)
reduced_embeddings = umap_model.fit_transform(embeddings) # for visualization (to avoid running encoding again)
# overall topic model pipeline
topic_model = BERTopic(
    embedding_model=embedding_model,          # Step 1 - Extract embeddings
    umap_model=umap_model,                    # Step 2 - Reduce dimensionality
    hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
    vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
    ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words)
    calculate_probabilities=True
)
topics, probs = topic_model.fit_transform(codes,  
                                        embeddings = embeddings) # return assignment of topics per doc

topic_info = topic_model.get_topic_info()
print(len(topic_info))
display(topic_info)


In [ ]:
# get representative examples: based on regex to code -- and taking the first match 
topic_info['Representative_Examples'] = topic_info['Representative_Docs'].apply(
    lambda x: {i: list(exp_df[exp_df['code']==i]['examples'])[0] for i in x})

topic_info['Representative_Examples'].head()

# save this out for LLM topic naming

In [ ]:
# Load in new topic names

import importlib
import pickle
import pandas as pd
# Load in LLM generated topic names
topic_info_new = []
with open('filepathtonewtopics', 'rb') as f:
    try: 
        while True:
            topic_info_new.append(pickle.load(f))
    except EOFError:
        pass

ti_new = pd.DataFrame(topic_info_new)
ti_new.columns = ti_new.iloc[0]
ti_new = ti_new[1:]

# reset index for processing
ti_new['new_index'] = range(len(ti_new))
ti_new = ti_new.set_index('new_index', drop=True)
sub_codes = sub_codes.reset_index(drop=True)
ti_new.shape

In [ ]:
# calculate coverage and relevance using embedding evaluation
from travail_code import evaluators as et
from travail_code import evaluators
importlib.reload(evaluators)
from travail_code import evaluators as et
from transformers import pipeline

threshold = .6
sim_model_id = "sentence-transformers/all-mpnet-base-v2"
sim_model = SentenceTransformer(sim_model_id)
# sim_model = embedding_model
sent_model_id = "siebert/sentiment-roberta-large-english"
sent_model= pipeline("sentiment-analysis",
                                model = sent_model_id)
evaluator = et.Evaluate_Pipeline(ti_new['LLM_namewex'], parent_codes, threshold,
                                        sim_model, sent_model = sent_model)
parent_results = evaluator()
subcode_results = evaluator.rerun_subcode(sub_codes)

score_keys = ['perc_coverage', 'perc_relevant_codes', 
            'avg_sim', 'std_sim', 
            'num_newcodes', 'total_output', 'total_gold']
      

In [ ]:
# get parent scores
{k: v for k,v in parent_results.items() if k in score_keys}

In [ ]:
# get subcode scores
{k: v for k,v in subcode_results.items() if k in score_keys}

In [ ]:
# get parent matches
parent_results['best_gold_tuple']